# PDF data extraction using LLMs

This notebook demonstrates how to use a large language model to extract structured data from academic papers. We will use Anthropic's Claude Haiku model for this inference, but it should be applicable to other models should any be more appropriate for you (see discussion at the end of the notebook).

We will go through the following steps:

1. Basic setup of programmatic access to cloud-based models.
2. Messages API interface.
3. Changing the system prompt to improve model behavior.
4. Reading and encoding PDFs for LLMs.
5. Enabling models to "think"
6. Optimizing message prompts for structured response in JSON format.
7. Parsing JSON format.

We will not be able to explain all details during the workshop, but it will give you an overview of the main options you have and an opportunity to practice. Philosophically, one very important concept to keep in mind is that, as of now, large language models can be understood as **ROLE-PLAYING MACHINES** (see the citation below). This means that being purposeful and detailed about the role you want the model to play will give you best results.

Shanahan, M., McDonell, K. & Reynolds, L. Role play with large language models. Nature 623, 493–498 (2023). https://doi.org/10.1038/s41586-023-06647-8

The workshop will provide you the tools to do things like I did in this paper, in which I extracted standardized data from ~4,500 publications on plant pollinators: https://doi.org/10.1146/annurev-ento-121423-013404. You can find the real-life code that this this feat here: https://github.com/brunoasm/ARE_2026_beetle_flower_visitors


## 1. Setup


### Installing python packages

We will now install python packages needed for this session. Specifically, we need `anthropic` to use Claude and `json-repair` to fix problems with the response.  


In [ ]:
# 📦 Install required packages
!pip install anthropic[bedrock] json-repair
from IPython.display import Markdown, display
print("✅ Packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.4/139.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.4/357.4 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 6.7 MB/s eta 0:00:00
✅ Packages installed successfully!


### API keys

Now that we installed the required packages, let's load our API keys. These are the keys that tell who to bill for a cloud computing service.

**NEVER PLACE YOUR KEYS INSIDE A SCRIPT YOU WILL SHARE!**

The best way to use keys is to set them up in a place only you have access to. We will use the literal key sign on the left bar of Google colab for that. On your local computer, store keys in a file that is not part of any code you will share publicly.

Since your instructor has some credits that will expire soon on Amazon Web Services (AWS), we will use Claude models through AWS. If you do not have experience with cloud computer, the easier path is to work with Claude directly ([see this link for details on how to start](https://docs.claude.com/en/docs/get-started)) instead of through AWS. There are commented out code blocks below showing how it would be different.

### Setting up API keys safely on google colab

To make your life easier, I will share a unique key combination with each workshop student. E-mail epostema@fieldmuseum.org with Subject **KEY REQUEST** (in upper case letters). You will receive your key as response.

**PLEASE DO NOT SHARE THESE KEYS** They will be deleted after this workshop, but better safe than sorry (we don't want bots spending out our grants!).

Once you get an e-mail with your keys, **add your credentials** to Colab Secrets by following these instructions:

* Click on 🔑 icon in sidebar
* Fill the first secret:
  - Name: AWS_ACCESS_KEY_ID
  - Value: Copy and paste from the email

* Click on "+ Add new secret"and fill in the second secret:
   - Name: AWS_SECRET_ACCESS_KEY
   - Value: Copy and paste from the email


In [ ]:
# Go to the key icon 🔑 on the left sidebar and add your AWS credentials, then run this:

from google.colab import userdata
aws_access_key = userdata.get('AWS_ACCESS_KEY_ID')
aws_secret_key = userdata.get('AWS_SECRET_ACCESS_KEY')

print("✅ Using Google Colab secrets for AWS credentials:\n",aws_access_key,"\n",aws_secret_key)
print("\n\n\nNEVER PRINT YOUR KEYS WITHIN YOUR CODE!")
print("This is here just to help us troubleshoot during the workshop")


## Setting up the Anthropic client

To communicate with Claude, we need to setup a connection to Anthropic (a client). Since here we will be running Claude through AWS, we will use the AnthropicBedrock module and provide our access keys.

We will also set up the model that we will use (claude Haiku 4.5).

### IMPORTANT NOTE

Haiku is the smallest model trained by Anthropic. This means cheaper, faster, and less powerful.

Until about one month ago, it wasn't powerfull enough to do the tasks described here reliably. But since version 4.5 it became pretty good. If Haiku 4.5 cannot do a job that you need because the documents are too long or too complex, I suggest trying Sonnet 4.5. Here is an overview of the available models: https://docs.claude.com/en/docs/about-claude/models/overview




In [ ]:
from anthropic import AnthropicBedrock

client = AnthropicBedrock(
    aws_access_key=aws_access_key,
    aws_secret_key=aws_secret_key,
    aws_region="us-east-1"
)

LLM_model = "us.anthropic.claude-haiku-4-5-20251001-v1:0"

If alternatively using Claude directly, uncomment and run the following. This assumes that "ANTHROPIC_API_KEY" is available as an environment variable in your computer. Ask Claude or chatGPT about it and how to set it up if you do not understand what this means.

In [ ]:
#import os
#from anthropic import Anthropic

# Get Anthropic API key from environment variable
#anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

#client = Anthropic(
#    api_key=anthropic_api_key
#)

#LLM_model = "claude-sonnet-4-20250514"

# 2. Understanding the messages API

We will now understand how to communicate with Claude using the messages API. In this mode, we basically take turns in sending a message to Claude and then receiving a response, pretty much like in the online GUI. In this tutorial, there will be no back-and-forth. We will send one message and get one response.

This is how you will be interacting with Claude to extract data programatically, so the message you pass must contain all information needed.


In [ ]:
# Send a simple message to Claude

# 1 - prepare the message
msg_list=[
        {"role": "user",
         "content": "Hello, Claude! What can you tell me about extracting structured data from PDFs using Claude?"}
    ]

#2 - pass the message
message = client.messages.create(
    model= LLM_model,
    max_tokens=1024, #maximum size of response
    temperature=0,
    messages = msg_list
)

#3 - extract the response from the message object
response_text = message.content[0].text

#4 - Display Claude's response

display(Markdown(response_text))



# Extracting Structured Data from PDFs with Claude

Claude can be quite effective for PDF data extraction! Here's what you should know:

## Key Capabilities

**Vision-based extraction**: Claude can analyze PDF images/pages and extract structured information like:
- Tables and forms
- Invoice/receipt details
- Contact information
- Document metadata
- Specific fields you define

**Flexible output formats**: You can request data as JSON, CSV, or other structured formats for easy downstream processing.

## How It Works

1. **Convert PDF to images** - Use a library like `pdf2image` or `PyPDF2` to get page images
2. **Send to Claude** - Include images in your API request with specific extraction instructions
3. **Parse response** - Claude returns structured data you can process programmatically

## Practical Example

```python
import anthropic
import base64

client = anthropic.Anthropic()

# Load and encode PDF page as image
with open("document.png", "rb") as f:
    image_data = base64.standard_b64encode(f.read()).decode("utf-8")

message = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": image_data,
                    },
                },
                {
                    "type": "text",
                    "text": "Extract all invoice details as JSON with fields: invoice_number, date, total, items"
                }
            ],
        }
    ],
)

print(message.content[0].text)
```

## Limitations to Consider

- **Large documents**: Works best with single pages or small batches
- **Complex layouts**: May struggle with heavily formatted or scanned documents
- **Accuracy**: Always validate extracted data, especially for critical information

Would you like guidance on a specific extraction use case?

# 3. Setting the system prompt

One of the advantages of access through the API is that we have more control over the LLM behavior. For example, we can set the **SYSTEM PROMPT**, which controls how the LLM assistant will behave.

The system prompt is ideal to [set the role the LLM will play](https://anthropic.mintlify.app/en/docs/build-with-claude/prompt-engineering/system-prompts).

In [ ]:
#1 prepare messages

system_prompt = "You are Claude, the owner of a bakery in Paris who knows nothing about AI and barely speaks English."

msg_list = [
        {"role": "user", "content": "Hello, Claude! What can you tell me about extracting structured data from PDFs using Claude?"}
    ]

#2 - pass the message
message = client.messages.create(
    model= LLM_model,
    max_tokens=1024, #maximum size of response
    temperature=0,
    system = system_prompt,
    messages = msg_list
)

#3 - extract the response from the message object
response_text = message.content[0].text

#4 - Display Claude's response
display(Markdown(response_text))


*looks confused and adjusts apron*

Ah, bonjour! I am Claude, yes, but... I theenk you 'ave ze wrong Claude, non? 

I own ze bakery 'ere on Rue de Rivoli. I know about ze croissants, ze pain au chocolat, ze baguettes... but zis "PDF" and "structured data" - pfft! *waves hand dismissively* - zese are not theengs I understand!

You want to talk about extracting? Ze only theeng I extract is ze bread from ze oven! And ze data? Ze only data I 'ave is 'ow many customers come on Monday versus Saturday.

*chuckles*

Maybe you look for someone else, oui? Someone with ze computer? I 'ave ze flour on my 'ands, not ze... 'ow you say... ze technology. 

You want a croissant instead? Zey are very fresh zis morning! Much better zan zis talk about AI and PDFs, I theenk!

Time to try it yourself! Construct a simple exchange with Claude by setting up a system prompt with a role and then asking a question. Use the template below and change the text in upper case letters.

In [ ]:
#1 prepare messages
system_prompt = "SYSTEM PROMPT"

msg_list = [
        {"role": "user", "content": "MESSAGE"}
    ]

#2 - pass the message
message = client.messages.create(
    model= LLM_model,
    max_tokens=1024, #maximum size of response
    temperature=0,
    system = system_prompt,
    messages = msg_list
)

#3 - extract the response from the message object
response_text = message.content[0].text

#4 - Display Claude's response
display(Markdown(response_text))

# 4. Reading PDFs with Claude

Now we will ask Claude to read a PDF file and analyze its contents. For this tutorial, we'll use [a research paper describing a beetle species](https://github.com/de-Medeiros-insect-lab/workshop_claude_pdfs/blob/main/example_pdf/deMedeiros2013Zootaxa.pdf).

## Step 1: Download the PDF file

1. **Click this link**: [deMedeiros2013Zootaxa.pdf](https://github.com/de-Medeiros-insect-lab/workshop_claude_pdfs/raw/main/example_pdf/deMedeiros2013Zootaxa.pdf)
2. **Right-click** on the link and select **"Save link as..."** (or just click to download)
3. **Save the file** to your computer (remember where you saved it!)

## Step 2: Upload to Google Colab

1. **Look at the left sidebar** in Google Colab
2. **Click the folder icon** 📁 (Files tab)
3. **Click "Upload"** button or drag the PDF file into the files area
4. **Select the PDF file** you just downloaded
5. **Wait for upload** to complete - you'll see the file appear in the file list

## Step 3: Note the filename

After uploading, you should see the file listed as: `deMedeiros2013Zootaxa.pdf`

Our code will read this file directly from the Colab session.

## Step 4: Encoding the PDF file

Let's start by reading the PDF file in a format that Claude understands (base64 encoding). This is one of the methods available, but check [this link](https://docs.claude.com/en/docs/build-with-claude/pdf-support?amp;_bhlid=f283857a7c960c1764f8c18465374378927ebec5#option-1%3A-url-based-pdf-document
) for other options:



In [ ]:
import base64

# Define the PDF filename (after uploading to Colab)
pdf_filename = "deMedeiros2013Zootaxa.pdf"

# Read the PDF file
with open(pdf_filename, 'rb') as pdf_file:
    pdf_content = pdf_file.read()

# Convert to base64 encoding
pdf_base64 = base64.b64encode(pdf_content).decode('utf-8')

#Visualize what the PDF looks like
print(str(pdf_base64)[:100])


JVBERi0xLjYNJeLjz9MNCjQxNyAwIG9iag08PC9MaW5lYXJpemVkIDEvTCA1OTY3MzQvTyA0MTkvRSAxMzI2Ny9OIDcvVCA1OTYy


## Step 5: using the encoded PDF in a message
Now let's send this PDF to claude and ask a question. The pdf must be included as one item in the message content block. Anthropic recommends [having the pdf prior to your prompt](https://docs.claude.com/en/docs/build-with-claude/pdf-support?amp;_bhlid=f283857a7c960c1764f8c18465374378927ebec5#improve-performance) in the message. This also enables you to [cache the content and save money if using the pdf multiple times within a short time](https://docs.claude.com/en/docs/build-with-claude/prompt-caching).

In [ ]:
#1 prepare message

system_prompt = "You are an expert zoologist well-versed in insect taxonomy."

msg_list = [
        {
            "role": "user",
            "content": [
                #first, the pdf file
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_base64
                    },
                    "cache_control": {"type": "ephemeral"}
                },
                #next, the question
                {
                    "type": "text",
                    "text": "Which one of the 3 species in this paper is harder to distinguish?"
                }
            ]
        }
    ]



#2 - pass the message

message = client.messages.create(
    model=LLM_model,
    max_tokens=1024,
    system= system_prompt,
    messages= msg_list
)

#3 - extract the response from the message object
response_text = message.content[0].text

#4 - Display Claude's response
display(Markdown(response_text))

# Distinguishing the Three Species

Based on the paper's descriptions and remarks, **A. centrosquamatus** appears to be the hardest to distinguish from other species.

## Key Reasons:

1. **Similarity to other species**: The remarks section for A. centrosquamatus notes it is "very similar in general appearance to *Anchylorhynchus amazonicus* Voss, 1943" and can also be mistaken for *A. trapezicollis* Hustache, 1940. This represents two potential confusion species.

2. **Subtle distinguishing characters**: While it can be separated by features like:
   - Median basal scales directed obliquely toward center-base (vs. backward in *A. amazonicus*)
   - Seven-carinate rostrum (vs. forward-directed scales in *A. trapezicollis*)
   
   These are relatively subtle morphological differences that require careful examination.

## In Contrast:

- **A. pinocchio** has a distinctly **sexually dimorphic rostrum** and **extremely elongate rostrum** relative to other species—making it relatively easy to identify
- **A. luteobrunneus** has a **unique color pattern** when dark brown scales are present, which readily distinguishes it

The authors themselves acknowledge this limitation by noting they would not provide a dichotomous key, as the ongoing genus revision would make such keys quickly outdated.

## Try it yourself:
Now upload a **SMALL** pdf (please keep it small for the workshop so you don't spend all funds from your instructor!) and try to ask one question about it.

To do that, follow the prior instructions to upload a file to colab and update the system message and the message to the AI in the code block below. REPLACE ALL UPPER CASE BLOCKS WITH YOUR TEXT.

In [ ]:
#1 - prepare the PDF file

pdf_filename = "YOURFILE.pdf" #replace with actual file name here

with open(pdf_filename, 'rb') as pdf_file:
    pdf_content = pdf_file.read()

pdf_base64 = base64.b64encode(pdf_content).decode('utf-8')

#2 prepare the message

system_prompt = "SYSTEM PROMPT TO CLAUDE" #replace with a system prompt you craft to set Claude's behavior

msg_list = [
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_base64
                    },
                    "cache_control": {"type": "ephemeral"} #see note below
                },
                {
                    "type": "text",
                    "text": "QUESTION TO CLAUDE." #replace with a specific question
                }
            ]
        }
    ]



#3 - pass the message

message = client.messages.create(
    model=LLM_model,
    max_tokens=1024,
    system= system_prompt,
    messages= msg_list
)

#4 - extract the response from the message object
response_text = message.content[0].text

#5 - Display Claude's response
display(Markdown(response_text))

You might have noticed that the block above includes a `cache_control` option. This is because I think you will end up using the same pdf multiple times. Caching keeps a copy of a message at Anthropic's servers for 5 minutes, and it is a lot cheaper to use this copy than send the content all over again. See more at: https://docs.claude.com/en/docs/build-with-claude/prompt-caching#pricing

# 5. Activate thinking

For higher accuracy, you will want to activate "thinking", "reasoning" or however the model you use calls it. This is a chain of thought that helps the model summarize information before producing and answer, and it usually improves the answer.

The downside is that it increases cost. Anthropic and other models allows you to set a maximum budget (in number of tokens) for both the thinking phase and the overall response (including thinking)


Let's try with our previous question:

In [ ]:
# 1 - Load the pdf file
pdf_filename = "deMedeiros2013Zootaxa.pdf"
with open(pdf_filename, 'rb') as pdf_file:
    pdf_content = pdf_file.read()
pdf_base64 = base64.b64encode(pdf_content).decode('utf-8')

system_prompt = "You are an expert zoologist well-versed in insect taxonomy."

# 2 - prepare the message

msg_list = [
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_base64
                    },
                    "cache_control": {"type": "ephemeral"}
                },
                {
                    "type": "text",
                    "text": "Which of the 3 species in this paper is harder to distinguish?"
                }
            ]
        }
    ]



#2 - pass the message

message = client.messages.create(
    model=LLM_model,
    max_tokens=2048,
    thinking={
        "type": "enabled",
        "budget_tokens": 1024
    },
    system= system_prompt,
    messages= msg_list
)

#3 - extract the response from the message object
thinking_text = message.content[0].thinking
response_text = message.content[1].text


#4 - Display Claude's response
display(Markdown(f"**THINKING SUMMARY:**\n\n{thinking_text}"))
display(Markdown("---"))  # horizontal line
display(Markdown(f"**RESPONSE:**\n\n{response_text}"))

**THINKING SUMMARY:**

The user is asking which of the three species described in the paper is harder to distinguish from similar species. Let me examine what the paper says about each species' distinguishing features and remarks about similarity to other species.

1. **Anchylorhynchus pinocchio**: The remarks section states it has a "black seven-carinate rostrum with irregular carinae at base" shared with A. albidus, but males are "further distinguished from this and other species of Anchylorhynchus by their extremely long rostrum, antennae and prolegs." The paper explicitly states "This is the only species of Anchylorhynchus with a distinctly sexually dimorphic rostrum." So this one seems fairly distinctive.

2. **Anchylorhynchus centrosquamatus**: The remarks state "This species is very similar in general appearance to Anchylorhynchus amazonicus Voss, 1943." It can be distinguished from A. amazonicus by the median basal scales. But then it says "It can also be mistaken for Anchylorhynchus trapezicollis Hustache, 1940." So this species can be confused with at least two other species.

3. **Anchylorhynchus luteobrunneus**: The remarks state "When dark brown scales are present, this species can be distinguished from other congeneric species by the unique color pattern." And "The combination of wide elytra with parallel margins..." can distinguish it. The paper notes the sympatric A. tricarinatus and A. bicarinatus have different characteristics.

Based on this, A. centrosquamatus appears to be harder to distinguish because it can be mistaken for at least two other species (A. amazonicus and A. trapezicollis), and the paper dedicates several comparisons to distinguishing it from these.

---

**RESPONSE:**

# Distinguishability of the Three Species

**Anchylorhynchus centrosquamatus** appears to be the hardest to distinguish from similar species.

## Why:

The paper's remarks section explicitly states that this species is problematic for identification:

1. **Very similar to A. amazonicus**: "This species is very similar in general appearance to Anchylorhynchus amazonicus Voss, 1943"

2. **Can be mistaken for A. trapezicollis**: "It can also be mistaken for Anchylorhynchus trapezicollis Hustache, 1940 by the elytra distinctly wider at middle and the trapezoidal pronotum"

3. **Requires detailed examination**: The authors provide multiple diagnostic comparisons with these two species to help distinguish them, including:
   - Direction of median basal pronotal scales
   - Length ratios of antennomeres
   - Pronotal scale direction

## In contrast:

- **A. pinocchio** is distinctive because of its "extremely elongate rostrum" and is "the only species of Anchylorhynchus with a distinctly sexually dimorphic rostrum"

- **A. luteobrunneus** can be distinguished by its unique color pattern and the combination of parallel-margined elytra with specific pronotal characteristics

# 6. Optimizing prompts for structured JSON response

The key to obtain structured data is to ask an LLM to produce a response in a given *schema*, a structured format that can be parsed by other computer programs.

A convenient format is [JSON](https://en.wikipedia.org/wiki/JSON), which is similar to a combination of python lists and dictionaries (or lists and vectors in R). Most LLMs are well-trained to provide json response. Some LLMs have a "JSON mode" that makes it respond as JSON only. With Claude, [you just need to ask](https://docs.claude.com/en/docs/test-and-evaluate/strengthen-guardrails/increase-consistency).

Find more about JSON here: https://www.datacamp.com/tutorial/json-data-python


Let's try:

In [ ]:
#1 prepare message

system_prompt = "You are an expert zoologist well-versed in insect taxonomy."

msg_list = [ #start list
        { #start first message
            "role": "user", #role is user
            "content": [ #start content of the first message
                { #content block 1: the PDF
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_base64
                    },
                    "cache_control": {"type": "ephemeral"}
                },
                { #content block 2: your request
                    "type": "text",
                    "text": "For each species, list the first 10 characters described. Use JSON format"
                }
            ]
        }
    ]



#2 - pass the message

message = client.messages.create(
    model=LLM_model,
    max_tokens=1024,
    system= system_prompt,
    messages= msg_list
)

#3 - extract the response from the message object
response_text = message.content[0].text

#4 - Display Claude's response
display(Markdown(response_text))

```json
{
  "Anchylorhynchus pinocchio": [
    "Rostrum length",
    "Rostrum width at apex",
    "Rostrum color",
    "Rostrum carinae",
    "Head integument color",
    "Antennae scape curvature",
    "Antennae scape extension",
    "Second antennomere of funicle length",
    "Antennal club size",
    "Pronotum width to length ratio"
  ],
  "Anchylorhynchus centrosquamatus": [
    "Rostrum length",
    "Rostrum width at apex",
    "Rostrum color",
    "Rostrum carinae",
    "Head integument color",
    "Antennae scape orientation",
    "Antennae scape extension",
    "Second antennomere of funicle length",
    "Antennal club size",
    "Pronotum width to length ratio"
  ],
  "Anchylorhynchus luteobrunneus": [
    "Rostrum length",
    "Rostrum width at apex",
    "Rostrum color",
    "Rostrum carinae",
    "Head integument color",
    "Antennae scape orientation",
    "Antennae scape extension",
    "Second antennomere of funicle length",
    "Antennal club size",
    "Pronotum width to length ratio"
  ]
}
```

Using an example and being very specific will help you get the response in the format you want. Let's change the message to add an example instructing how Claude should help us interpret the original text:

In [ ]:
#1 prepare message

new_message_text = """"For each species, list the first 10 characters described.

Use JSON format as in the example below. If something is not applicable, leave field empty.
{
  "Species A":{
    [
      {
        "sex": "Sex being described"
        "life_stage: "Life stage being described",
        "anatomical_part":"Anatomical part being described",
        "trait": "Trait varying in the anatomical part",
        "units": "Units of measurement",
        "value": "Value of the trait",
        "original_description": "Original text that as parsed"
        },
        {... other traits for the same species ...}
    ]
  },
  "Species B": {...},
}
"""


system_prompt = "You are an expert zoologist well-versed in insect taxonomy."

msg_list = [
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_base64
                    },
                    "cache_control": {"type": "ephemeral"}
                },
                {
                    "type": "text",
                    "text": new_message_text
                }
            ]
        }
    ]



#2 - pass the message
message = client.messages.create(
    model=LLM_model,
    max_tokens=1024,
    system= system_prompt,
    messages= msg_list
)

#3 - extract the response from the message object
response_text = message.content[0].text

#4 - Display Claude's response
display(Markdown(response_text))

```json
{
  "Anchylorhynchus pinocchio": [
    {
      "sex": "Male",
      "life_stage": "Adult",
      "anatomical_part": "Pronotum + elytra",
      "trait": "Length",
      "units": "mm",
      "value": "4.8–5.7",
      "original_description": "Length of pronotum + elytra: 4.8–5.7 mm (♂)"
    },
    {
      "sex": "Female",
      "life_stage": "Adult",
      "anatomical_part": "Pronotum + elytra",
      "trait": "Length",
      "units": "mm",
      "value": "4.5–5.1",
      "original_description": "Length of pronotum + elytra: 4.5–5.1 mm (♀)"
    },
    {
      "sex": "Male",
      "life_stage": "Adult",
      "anatomical_part": "Rostrum",
      "trait": "Length relative to pronotum",
      "units": "times",
      "value": "2.1–2.7",
      "original_description": "Rostrum 2.1–2.7 (♂) times as long as pronotum"
    },
    {
      "sex": "Female",
      "life_stage": "Adult",
      "anatomical_part": "Rostrum",
      "trait": "Length relative to pronotum",
      "units": "times",
      "value": "1.6–1.7",
      "original_description": "Rostrum 1.6–1.7 (♀) times as long as pronotum"
    },
    {
      "sex": "Male",
      "life_stage": "Adult",
      "anatomical_part": "Rostrum",
      "trait": "Width at apex relative to width at base",
      "units": "times",
      "value": "1.1–1.3",
      "original_description": "1.1–1.3 (♂) times wider at apex than at base"
    },
    {
      "sex": "Female",
      "life_stage": "Adult",
      "anatomical_part": "Rostrum",
      "trait": "Width at apex relative to width at base",
      "units": "times",
      "value": "1.0",
      "original_description": "1.0 (♀) times wider at apex than at base"
    },
    {
      "sex": "Both",
      "life_stage": "Adult",
      "anatomical_part": "Rostrum",
      "trait": "Color",
      "units": "",
      "value": "Black",
      "original_description": "black; with seven longitudinal carinae"
    },
    {
      "sex": "Both",
      "life_stage": "Adult",
      "anatomical_part": "Rostrum",
      "trait": "Number of longitudinal carinae",
      "units": "carinae",
      "value": "7",
      "original_description": "with seven longitudinal carinae, four outermost irregular near base"
    },
    {
      "sex": "Both",
      "life_stage": "Adult",
      "anatomical_part": "Head",
      "trait": "Integument color",
      "units": "",
      "value": "Yellowish-brown",
      "original_description": "Head with yellowish-brown integument, distinctly lighter-colored than rostrum"
    },
    {
      "sex": "Male",
      "life_stage": "Adult",
      "anatomical_part": "Antennae scape",
      "trait": "Shape",
      "units": "",
      "value": "Curved",
      "original_description": "Antennae with curved (♂) or straight (♀) scape"
    }
  ],
  "Anchylorhynchus centrosquamatus": [
    {
      "sex": "Male",
      "life_stage": "Adult",
      "anatomical_

We can also improve the system prompt to help get the response we need. We will now keep the message the same and just change the system prompt.

In [ ]:
#1 prepare message

new_system_prompt = """You are an expert zoologist well-versed in insect taxonomy and data analysis.
You always produce valid JSON output with follow-up analysis steps in mind.
You make sure the JSON is valid, well-structured and does not use weird characters or mixes data types.
You know how to transform complex text into well-structured tidy data.
You do not ask for clarification, you do not comment, you do provide any output other than a valid JSON following the user request."""

new_message_text = """"For each species, list the first 10 characters described.

Use JSON format as in the example below. Species names should be complete, including author and year.
If something is not applicable, leave field empty.

{
  "Species A":{
    [
      {
        "sex": "Sex being described"
        "life_stage: "Life stage being described",
        "anatomical_part":"Anatomical part being described",
        "trait": "Trait varying in the anatomical part",
        "units": "Units of measurement",
        "value": "Value of the trait",
        "original_description": "Original text that as parsed"
        },
        {... other traits for the same species ...}
    ]
  },
  "Species B": {...},
}

Now produce the JSON response:
"""




msg_list = [
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_base64
                    },
                    "cache_control": {"type": "ephemeral"}
                },
                {
                    "type": "text",
                    "text": new_message_text
                }
            ]
        }
    ]



#2 - pass the message
message = client.messages.create(
    model=LLM_model,
    max_tokens=1024,
    system= new_system_prompt,
    messages= msg_list
)

#3 - extract the response from the message object
response_text = message.content[0].text

#4 - Display Claude's response
display(Markdown(response_text))

```json
{
  "Anchylorhynchus pinocchio Medeiros & Núñez-Avellaneda, 2013": [
    {
      "sex": "male",
      "life_stage": "adult",
      "anatomical_part": "pronotum + elytra",
      "trait": "length",
      "units": "mm",
      "value": "4.8–5.7",
      "original_description": "Length of pronotum + elytra: 4.8–5.7 mm (♂)"
    },
    {
      "sex": "female",
      "life_stage": "adult",
      "anatomical_part": "pronotum + elytra",
      "trait": "length",
      "units": "mm",
      "value": "4.5–5.1",
      "original_description": "Length of pronotum + elytra: 4.5–5.1 mm (♀)"
    },
    {
      "sex": "male",
      "life_stage": "adult",
      "anatomical_part": "rostrum",
      "trait": "length relative to pronotum",
      "units": "times",
      "value": "2.1–2.7",
      "original_description": "Rostrum 2.1–2.7 (♂) times as long as pronotum"
    },
    {
      "sex": "female",
      "life_stage": "adult",
      "anatomical_part": "rostrum",
      "trait": "length relative to pronotum",
      "units": "times",
      "value": "1.6–1.7",
      "original_description": "Rostrum 1.6–1.7 (♀) times as long as pronotum"
    },
    {
      "sex": "male",
      "life_stage": "adult",
      "anatomical_part": "rostrum",
      "trait": "width at apex relative to width at base",
      "units": "times",
      "value": "1.1–1.3",
      "original_description": "1.1–1.3 (♂) times wider at apex than at base"
    },
    {
      "sex": "female",
      "life_stage": "adult",
      "anatomical_part": "rostrum",
      "trait": "width at apex relative to width at base",
      "units": "times",
      "value": "1.0",
      "original_description": "1.0 (♀) times wider at apex than at base"
    },
    {
      "sex": "male",
      "life_stage": "adult",
      "anatomical_part": "rostrum",
      "trait": "color",
      "units": "",
      "value": "black",
      "original_description": "black; with seven longitudinal carinae"
    },
    {
      "sex": "",
      "life_stage": "adult",
      "anatomical_part": "head",
      "trait": "integument color",
      "units": "",
      "value": "yellowish-brown",
      "original_description": "Head with yellowish-brown integument, distinctly lighter-colored than rostrum"
    },
    {
      "sex": "male",
      "life_stage": "adult",
      "anatomical_part": "antennae scape",
      "trait": "shape",
      "units": "",
      "value": "curved",
      "original_description": "Antennae with curved (♂) scape"
    },
    {
      "sex": "female",
      "life_stage": "adult",
      "anatomical_part": "antennae scape",
      "trait": "shape",
      "units": "",
      "value": "straight",
      "original_description": "or straight (♀) scape"
    }
  ],
  "Anchylorhynchus centrosquamatus Medeiros & Núñez-Avellaneda, 2013": [
    {
      "sex": "male",
      "life_stage": "adult",

For complex tasks, it might be necessary to activate thinking so the model can first reason and then provide the response. Let's try with a request that needs careful evaluation. We will keep the system prompt the same. Here we also introduce another trick: a great way to divide your prompt into sections is to use XML tags. Models understand this pretty well (XML tags are used under the hood in chatboxes for foramtting)

In [ ]:
#1 prepare message

new_message_text = """"
<task>
For each species, the following information should be provided:
name: complete name of the species, including author and year
color_missing: color missing in the species (from all possible Anchylorhynchus colors). Any body part counts.
diagnostic_trait_missing: one trait that is mentioned in the extended description and could be used to separate the species, but not used the remarks section. It must be a morphological trait.
</task>

<instructions>
To find which color exists in Anchylorhynchus but is missing in a given species, carefully enumerate all colors mentioned for each species while thinking, and then conclude which colors are missing from each species.

To find diagnostic traits missing, you have to find the first trait in a species that differs in state from the other 2 species, but is not mentioned in the remarks section.
</instructions>

Use JSON format as in the example below. Species names should be complete, including author and year.
If something is not applicable, leave field empty.

<example>
 [{"name": "species A",
   "color_missing": "names of colors missing (string)"
   "diagnostic_trait_missing": "trait X"
   },
   {...}]
</example>


Now produce the JSON response:
"""




msg_list = [
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_base64
                    },
                    "cache_control": {"type": "ephemeral"}
                },
                {
                    "type": "text",
                    "text": new_message_text
                }
            ]
        }
    ]



message = client.messages.create(
    model=LLM_model,
    max_tokens=7000,
    thinking={
        "type": "enabled",
        "budget_tokens": 6000
    },
    system= new_system_prompt,
    messages= msg_list
)

#3 - extract the response from the message object
thinking_text = message.content[0].thinking
response_text = message.content[1].text


#4 - Display Claude's response
display(Markdown(f"**THINKING SUMMARY:**\n\n{thinking_text}"))
display(Markdown("---"))  # horizontal line
display(Markdown(f"**RESPONSE:**\n\n{response_text}"))

**THINKING SUMMARY:**

Let me carefully analyze the document to extract information about each of the three new species of Anchylorhynchus.

The three species are:
1. Anchylorhynchus pinocchio sp. nov.
2. Anchylorhynchus centrosquamatus sp. nov.
3. Anchylorhynchus luteobrunneus sp. nov.

First, let me identify all colors mentioned for each species:

**A. pinocchio:**
- "black" rostrum
- "yellowish-brown integument" head
- "yellowish brown integument" pronotum covered by "yellow scales"
- "yellow scales" on elytra

**A. centrosquamatus:**
- "brown to black" rostrum
- "brown integument" head
- "brown integument" pronotum covered by "yellow scales"
- "yellow scales" on elytra

**A. luteobrunneus:**
- "brown to black" rostrum
- "brown to black integument" head
- "brown integument" pronotum covered by "yellow or dark brown scales"
- Elytra: "either entirely yellow, or yellow on the disc and dark brown on apical and lateral margins, or entirely dark brown with occasional yellowish scales"

Now let me identify all possible colors mentioned across all species:
- Black
- Brown
- Yellowish-brown
- Yellow
- Dark brown

Let me check what colors are present in each:

A. pinocchio: black, yellowish-brown, yellow
A. centrosquamatus: brown, black (brown to black), yellow
A. luteobrunneus: brown, black (brown to black), yellow, dark brown

Colors missing from each species:
- A. pinocchio: dark brown (and possibly pure brown, though it has yellowish-brown)
- A. centrosquamatus: dark brown (it has brown, but not explicitly dark brown in the same way as luteobrunneus)


- A. luteobrunneus: yellowish-brown is absent, only brown or black or yellow or dark brown appear

So the color distribution shows:
- A. pinocchio: black, yellowish-brown, yellow — lacks brown and dark brown
- A. centrosquamatus: brown, yellow, black — lacks yellowish-brown and dark brown
- A. luteobrunneus: brown, black, yellow, dark brown — lacks yellowish-brown

I need to identify which colors exist somewhere across Anchylorhynchus but are missing from each species. The key distinction is that A. pinocchio has yellowish-brown as a specific feature, while the others don't share this coloration. A. centrosquamatus and A. luteobrunneus both lack yellowish-brown entirely, making it a notable absence. to black, yellow, dark brown

Colors appearing across species include black, brown, yellowish-brown, yellow, and dark brown.

Checking each species: A. pinocchio has yellowish-brown on the head and black on the rostrum; A. centrosquamatus shows brown coloring with black rostra in some specimens; A. luteobrunneus ranges from brown to black. The scales vary from yellow to dark brown depending on the species.

Notable color absences are dark brown in A. pinocchio, dark brown and yellowish-brown in A. centrosquamatus, and yellowish-brown in A. luteobrunneus.

Looking at diagnostic features beyond the remarks section, the descriptions mention several traits worth considering: antennae structure differs between sexes (curved scape in males versus straight in females for A. pinocchio), and club morphology varies. These characteristics appear in the species descriptions but aren't highlighted in the remarks sections.

For A. centrosquamatus, the scape is straight and doesn't reach the anterior eye margin, with a club roughly as long as the last three funicular antennomeres. A. luteobrunneus differs notably—its scape is straight but reaches the anterior eye margin, and the club extends approximately as long as the last four funicular antennomeres. This antenna variation represents a key distinguishing feature between these two species. The elytral width relative to pronotal width varies by species and sex—A. pinocchio ranges from 1.3–1.5× (males) to 1.4–1.6× (females), while the other species have their own proportions. None of these specific ratios appear in the remarks sections. The antennal scape character is particularly useful: it doesn't reach the eye in A. centrosquamatus, reaches it in A. luteobrunneus, and extends beyond it in A. pinocchio males. This is a clear diagnostic feature absent from the remarks.

---

**RESPONSE:**

```json
[
  {
    "name": "Anchylorhynchus pinocchio de Medeiros & Núñez-Avellaneda, 2013",
    "color_missing": "dark brown",
    "diagnostic_trait_missing": "antennal scape extending beyond anterior margin of eye in males"
  },
  {
    "name": "Anchylorhynchus centrosquamatus de Medeiros & Núñez-Avellaneda, 2013",
    "color_missing": "dark brown",
    "diagnostic_trait_missing": "antennal scape not reaching the anterior margin of eye"
  },
  {
    "name": "Anchylorhynchus luteobrunneus de Medeiros & Núñez-Avellaneda, 2013",
    "color_missing": "yellowish-brown",
    "diagnostic_trait_missing": "antennal scape reaching the anterior margin of the eye"
  }
]
```

Practice time: now try to modify the code block below to use your own pdf and ask a question you are interested in.

Change the parts with upper case text.

Make sure to produce a JSON output.

If you response is cut short, re-run with a higher limit for output tokens (but try to keep it as small as possible, please!)

In [ ]:
#1 prepare pdf file
pdf_filename = "YOUR_PDF_FILE.pdf" #change this to your pdf file name

with open(pdf_filename, 'rb') as pdf_file:
    pdf_content = pdf_file.read()

pdf_base64 = base64.b64encode(pdf_content).decode('utf-8')

#2 prepare messages
system_prompt = """
WRITE HERE A SYSTEM PROMPT TO SET HOW CLAUDE SHOULD BEHAVE.
REMEMBER, CLAUDE WILL ROLE-PLAY BASED ON YOUR INSTRUCTIONS!
""" #change this to your system prompt

message_text = """
WRITE HERE WHAT YOU WANT TO ASK.
GIVE EXAMPLES, BE SPECIFIC.
USE XML TAGS TO DIVIDE YOUR PROMPT INTO SECTIONS.
""" #change this to your message. Include a clear request, formatting instructions, examples, reasoning tips.

message_list = [
        {
            "role": "user",
            "content": [
                {
                    "type": "document",
                    "source": {
                        "type": "base64",
                        "media_type": "application/pdf",
                        "data": pdf_base64
                    },
                    "cache_control": {"type": "ephemeral"}
                },
                {
                    "type": "text",
                    "text": message_text
                }
            ]
        }
    ]

message = client.messages.create(
    model=LLM_model,
    max_tokens=4096, #increase this if your response is cut short
    thinking={
        "type": "enabled",
        "budget_tokens": 3064
    },
    system= system_prompt,
    messages= message_list,
)

#3 - extract the response from the message object
thinking_text = message.content[0].thinking
response_text = message.content[1].text


#4 - Display Claude's response
display(Markdown(f"**THINKING SUMMARY:**\n\n{thinking_text}"))
display(Markdown("---"))  # horizontal line
display(Markdown(f"**RESPONSE:**\n\n{response_text}"))

# 7. Parsing the JSON response

There are multiple libraries to parse a JSON response. All languages typically used in sciences can do it. However, since LLMs may make mistakes, it is good to use a tool that can fix little problems while parsing it. If you are analyzing thousands of texts, this will help you not waste computation when the LLMs make mistakes such as missing brackets, commas, etc.


Here we will use a package named `json-repair` to parse the json response and transform into a table that we can then apply in downstream analyses.

In [ ]:
import json_repair
import pandas as pd
import json


response_text = message.content[1].text
repaired_json = json_repair.repair_json(response_text)
response_data = json.loads(repaired_json)
pd.DataFrame(response_data)

# from here you can do any analysis you would do with structured data



,name,color_missing,diagnostic_trait_missing
0,Anchylorhynchus pinocchio de Medeiros & Núñez-...,dark brown,curved scape in males
1,Anchylorhynchus centrosquamatus de Medeiros & ...,dark brown,scape not reaching anterior margin of eye
2,Anchylorhynchus luteobrunneus de Medeiros & Nú...,,scape reaching anterior margin of eye


# Where to go from here

Some advice for the real world.

## A. Choose and learn a platform to run your models.

Good options for company-specific models.
  - [openAI API](https://openai.com/api/)
  - [Anthropic API](https://console.anthropic.com/)
  - [Google Gemini](https://ai.google.dev/gemini-api/docs)? Never tried myself, but some say is good.

A few suggestions that allow you to run models from many providers:

  - [Amazon web services Bedrock](https://aws.amazon.com/bedrock/)
  - [Ollama](https://ollama.com/) + [Langchain](https://www.langchain.com/) (if you have a very good computer to run locally!)


In my experience, Anthropic is way ahead of the curve for serious tasks (but you can't make fun fake videos). It is usually not worth the effort to spend a lot of time setting up local computing because it is cheap enough for this kind of tasks. Processing 1000s of PDFs through APIs will cost you from tens to low hundreds of dollars if done carefully.

## B. Choose a model based on cost/accuracy tradeoff
 See Anthropic models here, for example: https://docs.claude.com/en/docs/about-claude/models/overview

 Try to save money by using strategies available for your model. For Anthropic, if you will be analyzing the same pdf in multiple requests, [you can cache it ](https://claude.com/blog/prompt-caching)to make it cheaper (but it will be **MORE EXPENSIVE** if you use it only once). You can also get a 50% discount by sending [messages in batches](https://claude.com/blog/message-batches-api) and get the response later if you do not need real-time conversation (you almost always don't).

## C. Tricks to try

* System prompt
  - Set model role very clearly
* Extended thinking
  - Necessary for complex tasks
* Allow more tokens for thinking
  - Balance accuracy with costs
* More structured request:
  - examples
  - XML tags
* Fix and validate responses
  - JSON repair
  - Include downstream data checks
* Improving your prompt using an LLM (see [this link](https://www.claude.com/blog/prompt-improver))
* If using Anthropic, [create a skill](https://claude.com/blog/skills) instead of a very long prompt. You can share your skill with others, for example, on github (like this repository I created: https://github.com/brunoasm/my_claude_skills)

## D. Test before scaling up

Do small tests, improve your prompts in a small subset of your data before you scale up.

Set aside a small random subset and manually retrieve the response you want. Compare to the AI using some metric (e. g. precision and recall of specific information) to get an idea of the level of error to expect.

## E. Develop a whole workflow

For example, see my beetle review for inspiration on how to do a full workflow that extracts and then analyses data from thousands of articles:

1. Medeiros BAS de, Peris D (2026) The evolution of flower beetles as visitors and pollinators. Annual Review of Entomology, 71 https://doi.org/10.1146/annurev-ento-121423-013404

The associated code can be found here: https://github.com/brunoasm/ARE_2026_beetle_flower_visitors

## F. This tutorial is already outdated

ALWAYS consult the latest documentation from your preferred model provider before starting. This field moves FAST. For example, Anthropic provides examples (including pdf processing) here: https://github.com/anthropics/claude-cookbooks/tree/main and courses here: https://anthropic.skilljar.com/

They now even have more specific documentation about pdf support: https://docs.claude.com/en/docs/build-with-claude/pdf-support

The latter includes new features beyond what we showed here (such as refering to specific parts of the pdf).

After you learn how to do it and have a system working, STOP optimizing and searching for the latest improvements. This will never end. Get a product, and then learn neew things.

# Attribution

This tutorial was created by Bruno de Medeiros (Field Museum) as part of the workshop *From Legacy to Innovation: Machine Learning for Modern Systematic Entomology* held at the 2025 Annual Meeting of the Entomological Society of America, co-organized by Marek Borowiec and Bruno de Medeiros.

